In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

# Memuat dataset
df = pd.read_csv('drive/MyDrive/komdigi_hoaks.csv')

# Melihat data teratas
print(df.head())
print(df.info())

      id                                                url  \
0  64168  https://www.komdigi.go.id/berita/berita-hoaks/...   
1  64167  https://www.komdigi.go.id/berita/berita-hoaks/...   
2  64166  https://www.komdigi.go.id/berita/berita-hoaks/...   
3  64165  https://www.komdigi.go.id/berita/berita-hoaks/...   
4  64163  https://www.komdigi.go.id/berita/berita-hoaks/...   

                                               title  \
0   [HOAKS] Rupiah Sengaja Dilemahkan Bank Indonesia   
1  [HOAKS] Bantuan Dana Pensiun 2026 Mengatasnama...   
2          [HOAKS] Tarif Listrik Naik Dua Kali Lipat   
3  [HOAKS] Tautan Pendaftaran Lowongan Telkom Ind...   
4  [HOAKS] Artikel Dadan Hindayana Sebut Jokowi T...   

                                                slug         published_at  \
0     hoaks-rupiah-sengaja-dilemahkan-bank-indonesia  2026-06-06 23:06:00   
1  hoaks-bantuan-dana-pensiun-2026-mengatasnamaka...  2026-06-06 23:04:00   
2            hoaks-tarif-listrik-naik-dua-kali-lipat 

In [4]:
# Memeriksa jumlah data kosong di setiap kolom
print(df.isnull().sum())

# Menghapus baris yang teks utamanya kosong
df = df.dropna(subset=['title', 'body_text'])

# Menampilkan 5 baris yang kolom 'main_image_url'-nya kosong
baris_kosong_gambar = df[df['main_image_url'].isnull()]
print(baris_kosong_gambar[['id', 'title', 'main_image_url']].head())

id                   0
url                  0
title                0
slug                 0
published_at         0
view_count           0
excerpt           3188
body_html            0
body_text            0
main_image_url       5
category             0
tags              2549
topics               0
dtype: int64
         id                                              title main_image_url
2701  57788  [HOAKS] Parfum, Pengharum Ruangan, dan Wangi D...            NaN
2791  57500  [HOAKS] TKA Cina Didatangkan untuk Jadi Operat...            NaN
2800  57459  [HOAKS] Surat Undangan Permohonan Bantuan Meng...            NaN
4179     61  [HOAKS] Lowongan Pekerjaan Pegawai di PT Djaru...            NaN
4211      8  [HOAKS] Undian Berhadiah Mengatasnamakan Found...            NaN


In [5]:
import re

def clean_text(text):
    # 1. Mengubah teks menjadi huruf kecil semua (Lowercasing)
    text = text.lower()

    # 2. Menghapus tag HTML (jika masih terbawa di body_text)
    text = re.sub(r'<.*?>', '', text)

    # 3. Menghapus URL/Link
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # 4. Menghapus karakter khusus, angka, dan tanda baca
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 5. Menghapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Menerapkan fungsi pembersihan ke kolom title dan body_text
df['clean_title'] = df['title'].apply(clean_text)
df['clean_body'] = df['body_text'].apply(clean_text)

In [6]:
pip install sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 9.0 MB/s eta 0:00:00


In [7]:
# Menghapus kata umum/stopwords
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

factory = StopWordRemoverFactory()
stopword_remover = factory.create_stop_word_remover()

def remove_stopwords(text):
    return stopword_remover.remove(text)

df['clean_body'] = df['clean_body'].apply(remove_stopwords)

In [8]:
# Stemming
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

factory_stem = StemmerFactory()
stemmer = factory_stem.create_stemmer()

df['stemmed_body'] = df['clean_body'].apply(lambda x: stemmer.stem(x))

In [9]:
# Memilih kolom-kolom yang sudah bersih saja untuk disimpan
df_ready = df[['id', 'clean_title', 'stemmed_body', 'category']]

# Menyimpan ke file baru
df_ready.to_csv('drive/MyDrive/komdigi_hoaks_clean.csv', index=False)
print("Preprocessing selesai! Data disimpan ke 'drive/MyDrive/komdigi_hoaks_clean.csv'")

Preprocessing selesai! Data disimpan ke 'drive/MyDrive/komdigi_hoaks_clean.csv'


In [10]:
# Hasil pre-processing, menampilkan data teratas
print(df_ready.head())
print(df_ready.info())

      id                                        clean_title  \
0  64168     hoaks rupiah sengaja dilemahkan bank indonesia   
1  64167  hoaks bantuan dana pensiun mengatasnamakan kem...   
2  64166            hoaks tarif listrik naik dua kali lipat   
3  64165  hoaks tautan pendaftaran lowongan telkom indon...   
4  64163  hoaks artikel dadan hindayana sebut jokowi ter...   

                                        stemmed_body           category  
0  jelas edar buah unggah video media sosial face...  Klarifikasi Hoaks  
1  jelas edar buah unggah video media sosial tikt...  Klarifikasi Hoaks  
2  jelas edar buah unggah media sosial facebook n...  Klarifikasi Hoaks  
3  jelas edar buah unggah media sosial threads kl...  Klarifikasi Hoaks  
4  jelas edar unggah tangkap layar judul artikel ...  Klarifikasi Hoaks  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4212 entries, 0 to 4211
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        ----------

In [11]:
df_ready['text'] = (
    df_ready['clean_title'].fillna('') +
    " " +
    df_ready['stemmed_body'].fillna('')
)

print(df_ready[['text', 'category']].head())

                                                text           category
0  hoaks rupiah sengaja dilemahkan bank indonesia...  Klarifikasi Hoaks
1  hoaks bantuan dana pensiun mengatasnamakan kem...  Klarifikasi Hoaks
2  hoaks tarif listrik naik dua kali lipat jelas ...  Klarifikasi Hoaks
3  hoaks tautan pendaftaran lowongan telkom indon...  Klarifikasi Hoaks
4  hoaks artikel dadan hindayana sebut jokowi ter...  Klarifikasi Hoaks


/tmp/ipykernel_403/802867467.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ready['text'] = (


# **PENGOLAHAN DATASET**

In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [13]:
# 1. Menentukan input (X) dan target label (y)
# 'stemmed_body' adalah kolom teks bersih hasil preprocessing sebelumnya
# 'topics' atau 'category' digunakan sebagai kolom target prediksi
X = df['stemmed_body']
y = df['topics'] # menyesuaikan kolom target

In [14]:
# 2. Membagi data menjadi Data training dan Data testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
# 3. Proses Ekstraksi menggunakan TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Maksimal 5000 kata terpenting
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)
# --- SIMULASI DATA (Gunakan kolom 'stemmed_body' dari dataset Anda) ---
# Kita ambil sampel 5 baris pertama agar tampilan hasilnya mudah dibaca dan tidak terlalu panjang
sampel_teks = df['stemmed_body'].head(5)

# 1. Inisialisasi TF-IDF Vectorizer
# Kita batasi max_features=15 kata terpenting agar tabelnya tidak terlalu lebar saat ditampilkan
tfidf_vectorizer = TfidfVectorizer(max_features=15)

# 2. Proses Ekstraksi (Fit & Transform)
tfidf_matrix = tfidf_vectorizer.fit_transform(sampel_teks)


# =====================================================================
# KODE TAMBAHAN UNTUK MENAMPILKAN HASIL EKSTRAKSI TF-IDF
# =====================================================================

# A. Menampilkan Daftar Kata Unik (Vocabulary/Fitur) yang Ditemukan
fitur_kata = tfidf_vectorizer.get_feature_names_out()
print("=== 1. DAFTAR KATA KUNCI (FITUR) YANG BERHASIL DIEKSTRAK ===")
print(fitur_kata)
print("\n" + "="*60 + "\n")

# B. Mengubah Matriks TF-IDF menjadi Tabel DataFrame Agar Mudah Dibaca
df_hasil_tfidf = pd.DataFrame(data=tfidf_matrix.toarray(), columns=fitur_kata)

# Menambahkan kolom indeks dokumen untuk memperjelas barisnya
df_hasil_tfidf.insert(0, 'Dokumen_Ke', range(1, len(df_hasil_tfidf) + 1))

print("=== 2. TABEL MATRIKS BOBOT ANGKA TF-IDF ===")
print(df_hasil_tfidf.to_string(index=False))

=== 1. DAFTAR KATA KUNCI (FITUR) YANG BERHASIL DIEKSTRAK ===
['bank' 'benar' 'edar' 'fakta' 'indonesia' 'jelas' 'klaim' 'laku' 'lansir'
 'link' 'rupiah' 'sebut' 'sosial' 'tidak' 'unggah']


=== 2. TABEL MATRIKS BOBOT ANGKA TF-IDF ===
 Dokumen_Ke     bank    benar     edar    fakta  indonesia    jelas    klaim     laku   lansir     link   rupiah    sebut   sosial    tidak   unggah
          1 0.556429 0.106057 0.053028 0.053028   0.372647 0.053028 0.106057 0.188089 0.053028 0.053028 0.667715 0.159085 0.053028 0.053028 0.053028
          2 0.000000 0.359354 0.179677 0.179677   0.252529 0.179677 0.179677 0.000000 0.179677 0.179677 0.000000 0.179677 0.179677 0.718707 0.179677
          3 0.000000 0.202453 0.202453 0.202453   0.000000 0.202453 0.404906 0.239364 0.202453 0.202453 0.000000 0.404906 0.404906 0.404906 0.202453
          4 0.000000 0.194997 0.194997 0.194997   0.548124 0.194997 0.389995 0.230550 0.194997 0.194997 0.000000 0.194997 0.194997 0.194997 0.389995
          5 0.000000 

In [16]:
# 4. Pemrosesan menggunakan Logistic Regression / SVM
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [17]:
# 5. Hasil Pengolahan Data
y_pred = model.predict(X_test_tfidf)
print("Akurasi Model:", accuracy_score(y_test, y_pred))
print("\nLaporan Klasifikasi:\n", classification_report(y_test, y_pred))

Akurasi Model: 0.8457888493475683

Laporan Klasifikasi:
                     precision    recall  f1-score   support

             Hoaks       0.85      0.99      0.92       715
         Kesehatan       0.00      0.00      0.00         9
Masyarakat Digital       0.00      0.00      0.00        17
    Pejabat Publik       0.50      0.08      0.13        26
          Penipuan       0.38      0.05      0.08        63
 Program/Kebijakan       0.00      0.00      0.00        13

          accuracy                           0.85       843
         macro avg       0.29      0.19      0.19       843
      weighted avg       0.77      0.85      0.79       843



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Kesimpulan
Konversi teks dari TF-IDF menghasilkan matriks berdimensi sangat tinggi, model SVM dan Logistic Regression sangat efektif dalam menangani jenis data spasial renggang (sparse matrix) tanpa mudah mengalami gejala overfitting. Sehingga menciptakan sistem pengolahan data yang ringan namun tetap menghasilkan akurasi yang sangat tinggi dalam mengklasifikasikan jenis-jenis hoaks dari dataset komdigi.

# **PENGOLAHAN DATASET MENGGUNAKAN MODEL LAIN**

### **SUPERVISED LEARNING**

**Decision tree**

In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Menentukan Variabel Input (X) dan Target (y)
X = df['stemmed_body']
y = df['topics'] # Menargetkan kolom topics hoaks

# 2. Membagi Data menjadi Data Latih (80%) dan Data Uji (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Ekstraksi Fitur dengan TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# 4. Inisialisasi dan Pelatihan Model Decision Tree
# Penjelasan parameter penting:
# - max_depth=20: Membatasi kedalaman pohon agar tidak bercabang terlalu ekstrem (mencegah overfitting)
# - criterion='gini': Indikator untuk mengukur kualitas pemisahan cabang (bisa juga menggunakan 'entropy')
model_dt = DecisionTreeClassifier(max_depth=20, criterion='gini', random_state=42)
model_dt.fit(X_train_tfidf, y_train)

# 5. Prediksi dan Evaluasi Model
y_pred_dt = model_dt.predict(X_test_tfidf)

print("=== HASIL EVALUASI MODEL DECISION TREE ===")
print("Akurasi Model:", accuracy_score(y_test, y_pred_dt))
print("\nLaporan Klasifikasi:\n", classification_report(y_test, y_pred_dt))

=== HASIL EVALUASI MODEL DECISION TREE ===
Akurasi Model: 0.8185053380782918

Laporan Klasifikasi:
                     precision    recall  f1-score   support

             Hoaks       0.88      0.91      0.90       715
         Kesehatan       0.12      0.11      0.12         9
Masyarakat Digital       0.25      0.12      0.16        17
    Pejabat Publik       0.24      0.19      0.21        26
          Penipuan       0.42      0.41      0.42        63
 Program/Kebijakan       0.50      0.15      0.24        13

          accuracy                           0.82       843
         macro avg       0.40      0.32      0.34       843
      weighted avg       0.80      0.82      0.81       843



**Random forest**

In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Menentukan Variabel Input (X) dan Target (y)
X = df['stemmed_body']
y = df['topics'] # Menargetkan kolom topics hoaks

# 2. Membagi Data menjadi Data Latih (80%) dan Data Uji (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Ekstraksi Fitur dengan TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# 4. Inisialisasi dan Pelatihan Model Random Forest
# Penjelasan parameter penting:
# - n_estimators=100: Membuat 100 pohon keputusan independen di dalam "hutan"
# - max_depth=20: Membatasi kedalaman setiap pohon agar komputasi tidak terlalu berat
# - n_jobs=-1: Menggunakan semua inti CPU komputer agar proses pelatihan lebih cepat
model_rf = RandomForestClassifier(n_estimators=100, max_depth=20, n_jobs=-1, random_state=42)
model_rf.fit(X_train_tfidf, y_train)

# 5. Prediksi dan Evaluasi Model
y_pred_rf = model_rf.predict(X_test_tfidf)

print("=== HASIL EVALUASI MODEL RANDOM FOREST ===")
print("Akurasi Model:", accuracy_score(y_test, y_pred_rf))
print("\nLaporan Klasifikasi:\n", classification_report(y_test, y_pred_rf))

=== HASIL EVALUASI MODEL RANDOM FOREST ===
Akurasi Model: 0.8481613285883749

Laporan Klasifikasi:
                     precision    recall  f1-score   support

             Hoaks       0.85      1.00      0.92       715
         Kesehatan       0.00      0.00      0.00         9
Masyarakat Digital       0.00      0.00      0.00        17
    Pejabat Publik       0.00      0.00      0.00        26
          Penipuan       0.00      0.00      0.00        63
 Program/Kebijakan       0.00      0.00      0.00        13

          accuracy                           0.85       843
         macro avg       0.14      0.17      0.15       843
      weighted avg       0.72      0.85      0.78       843



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


**Naive bayes**

In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

# 1. Menentukan Variabel Input (X) dan Target (y)
X = df['stemmed_body']
y = df['topics'] # Menargetkan kolom topics hoaks

# 2. Membagi Data menjadi Data Latih (80%) dan Data Uji (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Ekstraksi Fitur dengan TF-IDF
# Naive Bayes bekerja sangat baik bahkan jika jumlah kata kunci (max_features) sangat besar
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# 4. Inisialisasi dan Pelatihan Model Naive Bayes
# Parameter alpha=1.0 adalah Laplace Smoothing untuk mencegah probabilitas bernilai 0
# jika ada kata baru yang belum pernah muncul di data latihan.
model_nb = MultinomialNB(alpha=1.0)
model_nb.fit(X_train_tfidf, y_train)

# 5. Prediksi dan Evaluasi Model
y_pred_nb = model_nb.predict(X_test_tfidf)

print("=== HASIL EVALUASI MODEL NAIVE BAYES ===")
print("Akurasi Model:", accuracy_score(y_test, y_pred_nb))
print("\nLaporan Klasifikasi:\n", classification_report(y_test, y_pred_nb))

=== HASIL EVALUASI MODEL NAIVE BAYES ===
Akurasi Model: 0.8481613285883749

Laporan Klasifikasi:
                     precision    recall  f1-score   support

             Hoaks       0.85      1.00      0.92       715
         Kesehatan       0.00      0.00      0.00         9
Masyarakat Digital       0.00      0.00      0.00        17
    Pejabat Publik       0.00      0.00      0.00        26
          Penipuan       0.00      0.00      0.00        63
 Program/Kebijakan       0.00      0.00      0.00        13

          accuracy                           0.85       843
         macro avg       0.14      0.17      0.15       843
      weighted avg       0.72      0.85      0.78       843



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### **UNSUPERVISED LEARNING**

**K-Means clustering**

In [21]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

# 1. Menentukan Variabel Input (Hanya teks, tidak butuh target 'y')
X = df['body_text']

# 2. Ekstraksi Fitur dengan TF-IDF
# Untuk clustering, membatasi max_features membantu membuat hasil cluster lebih fokus
tfidf_vectorizer = TfidfVectorizer(max_features=1000)
X_tfidf = tfidf_vectorizer.fit_transform(X)

# 3. Inisialisasi dan Pelatihan Model K-Means
# Contoh: Kita ingin mengelompokkan data hoaks ini menjadi 5 tren/topik utama (n_clusters=5)
num_clusters = 5
model_kmeans = KMeans(n_clusters=num_clusters, init='k-means++', max_iter=300, random_state=42)
model_kmeans.fit(X_tfidf)

# 4. Memasukkan Hasil Cluster Kembali ke DataFrame Asli
df['cluster_label'] = model_kmeans.labels_


# =====================================================================
# KODE TAMBAHAN UNTUK MENAMPILKAN KATA KUNCI PER CLUSTER
# =====================================================================
print("=== HASIL K-MEANS CLUSTERING (KATA KUNCI TIAP KELOMPOK) ===\n")

# Mendapatkan koordinat pusat cluster (centroids) dan daftar kata dari TF-IDF
centroids = model_kmeans.cluster_centers_.argsort()[:, ::-1]
fitur_kata = tfidf_vectorizer.get_feature_names_out()

for i in range(num_clusters):
    print(f"--- Cluster {i+1} ---")
    # Mengambil 8 kata terpenting yang paling mendominasi cluster tersebut
    top_words = [fitur_kata[ind] for ind in centroids[i, :8]]
    print("Kata Kunci Dominan:", ", ".join(top_words))

    # Menampilkan jumlah dokumen yang masuk ke cluster ini
    jumlah_doc = len(df[df['cluster_label'] == i])
    print(f"Jumlah Berita Hoaks: {jumlah_doc} dokumen\n")

=== HASIL K-MEANS CLUSTERING (KATA KUNCI TIAP KELOMPOK) ===

--- Cluster 1 ---
Kata Kunci Dominan: yang, di, dan, tersebut, video, tidak, com, dari
Jumlah Berita Hoaks: 1740 dokumen

--- Cluster 2 ---
Kata Kunci Dominan: tautan, lowongan, pendaftaran, resmi, bank, yang, id, informasi
Jumlah Berita Hoaks: 792 dokumen

--- Cluster 3 ---
Kata Kunci Dominan: video, prabowo, yang, presiden, di, tersebut, com, youtube
Jumlah Berita Hoaks: 819 dokumen

--- Cluster 4 ---
Kata Kunci Dominan: akun, whatsapp, mengatasnamakan, nomor, kabupaten, penipuan, tersebut, dan
Jumlah Berita Hoaks: 643 dokumen

--- Cluster 5 ---
Kata Kunci Dominan: jokowi, presiden, video, joko, widodo, yang, artikel, ijazah
Jumlah Berita Hoaks: 218 dokumen



Logistic regression / SVM = 0,845
Decision tree = 0,81
Random forest = 0,848
Naive bayes = 0,848